In [1]:


import os
import shutil
import kagglehub

# 1. Download dataset using kagglehub
cache_path = kagglehub.dataset_download("blastchar/telco-customer-churn")

# 2. Ensure target directory exists
target_dir = "data/raw"
os.makedirs(target_dir, exist_ok=True)

# 3. Copy downloaded files from cache to project's data/raw folder
for file_name in os.listdir(cache_path):
    src_file = os.path.join(cache_path, file_name)
    dst_file = os.path.join(target_dir, file_name)
    if os.path.isfile(src_file):
        shutil.copy(src_file, dst_file)

print(f"Dataset downloaded and placed at: {os.path.abspath(target_dir)}")
print("Files inside data/raw:", os.listdir(target_dir))

✅ Dataset downloaded and placed at: C:\Users\waghm\MY PROJECTS\Telco Churn Predictor\data\raw
📂 Files inside data/raw: ['WA_Fn-UseC_-Telco-Customer-Churn.csv']


In [2]:
import os

old_path = "data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv"
new_path = "data/raw/telco_churn.csv"

# Check if the original file exists before renaming
if os.path.exists(old_path):
    os.rename(old_path, new_path)
    print(f"File successfully renamed to: {new_path}")
else:
    print("Original file not found (it might already have been renamed).")

# Verify the updated contents of data/raw
print("Files inside data/raw:", os.listdir("data/raw"))

✅ File successfully renamed to: data/raw/telco_churn.csv
📂 Files inside data/raw: ['telco_churn.csv']


In [3]:
from pathlib import Path

def display_tree(dir_path: Path, prefix: str = "", ignore: set = None):
    if ignore is None:
        ignore = {".git", "__pycache__", ".ipynb_checkpoints", ".pytest_cache", "venv"}
    
    contents = [p for p in dir_path.iterdir() if p.name not in ignore]
    contents.sort(key=lambda p: (p.is_file(), p.name.lower()))
    
    pointers = ["├── "] * (len(contents) - 1) + ["└── "]
    for pointer, path in zip(pointers, contents):
        print(f"{prefix}{pointer}{path.name}")
        if path.is_dir():
            extension = "│   " if pointer == "├── " else "    "
            display_tree(path, prefix=prefix + extension, ignore=ignore)

print(f"📁 {Path.cwd().name}/")
display_tree(Path.cwd())

📁 Telco Churn Predictor/
├── .github
│   └── workflows
├── artifacts
├── configs
├── data
│   ├── external
│   ├── processed
│   └── raw
│       └── telco_churn.csv
├── docker
├── great_expectations
├── mlruns
│   ├── .trash
│   ├── 225977856873923587
│   │   ├── 7691d96668ba40b49a6098eec048a233
│   │   │   ├── artifacts
│   │   │   ├── metrics
│   │   │   │   ├── f1
│   │   │   │   ├── precision
│   │   │   │   ├── pred_time_sec
│   │   │   │   ├── recall
│   │   │   │   ├── roc_auc
│   │   │   │   └── train_time_sec
│   │   │   ├── outputs
│   │   │   │   └── m-5c28683daae445cea808b85abbe9c47e
│   │   │   │       └── meta.yaml
│   │   │   ├── params
│   │   │   │   ├── colsample_bytree
│   │   │   │   ├── eval_metric
│   │   │   │   ├── gamma
│   │   │   │   ├── learning_rate
│   │   │   │   ├── max_depth
│   │   │   │   ├── min_child_weight
│   │   │   │   ├── n_estimators
│   │   │   │   ├── n_jobs
│   │   │   │   ├── prediction_threshold
│   │   │   │   ├── random_state
│   │   │ 

In [2]:
from pathlib import Path

required_files = [
    "data/raw/telco_churn.csv",
    "src/data/load_data.py",
    "src/data/preprocess_data.py",
    "src/features/build_features.py",
    "src/models/metrics.py",
    "src/models/tune.py",
    "src/models/train.py",
    "src/utils/tracking.py",
    "src/serving/predict.py",
]

missing = [f for f in required_files if not Path(f).exists()]
if missing:
    print(f"❌ Missing required files: {missing}")
else:
    print("✅ Project structure is complete and verified.")

✅ Project structure is complete and verified.


In [1]:
from pathlib import Path

# In Python scripts: uses __file__ | In Jupyter: falls back to current working directory
current_path = Path(__file__).resolve() if "__file__" in globals() else Path.cwd().resolve()

print(f"Current Path: {current_path}")
print(f"parents[0]:   {current_path.parents[0]}")
print(f"parents[1]:   {current_path.parents[1]}")
print(f"parents[2]:   {current_path.parents[2]}")

Current Path: C:\Users\waghm\MY PROJECTS\Telco Churn Predictor
parents[0]:   C:\Users\waghm\MY PROJECTS
parents[1]:   C:\Users\waghm
parents[2]:   C:\Users


In [2]:
%%writefile .dockerignore
.git
.gitignore
__pycache__
*.pyc
*.pyo
*.pyd
.pytest_cache
.venv
env/
venv/
.github

Overwriting .dockerignore


In [3]:
%%writefile Dockerfile
FROM python:3.11-slim AS base

ENV PYTHONUNBUFFERED=1 \
    PYTHONDONTWRITEBYTECODE=1 \
    MLFLOW_ALLOW_FILE_STORE=true

WORKDIR /app

RUN apt-get update && apt-get install -y --no-install-recommends \
    build-essential \
    curl \
    && rm -rf /var/lib/apt/lists/*

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY . .

# --- Target 1: FastAPI Backend ---
FROM base AS fastapi
EXPOSE 8000
CMD ["uvicorn", "src.app.main:app", "--host", "0.0.0.0", "--port", "8000"]

# --- Target 2: Streamlit Frontend ---
FROM base AS streamlit
EXPOSE 8501
CMD ["streamlit", "run", "app.py", "--server.port=8501", "--server.address=0.0.0.0"]

Overwriting Dockerfile


In [4]:
%%writefile docker-compose.yml
version: '3.8'

services:
  fastapi:
    build:
      context: .
      target: fastapi
    container_name: telco_fastapi_backend
    ports:
      - "8000:8000"
    volumes:
      - ./mlruns:/app/mlruns
    environment:
      - MLFLOW_ALLOW_FILE_STORE=true
    healthcheck:
      test: ["CMD", "curl", "-f", "http://localhost:8000/health"]
      interval: 10s
      timeout: 5s
      retries: 3

  streamlit:
    build:
      context: .
      target: streamlit
    container_name: telco_streamlit_frontend
    ports:
      - "8501:8501"
    environment:
      - FASTAPI_URL=http://fastapi:8000
    depends_on:
      fastapi:
        condition: service_healthy

Overwriting docker-compose.yml


In [5]:
%%writefile app.py
import os
import requests
import streamlit as st

FASTAPI_URL = os.getenv("FASTAPI_URL", "http://localhost:8000")

st.set_page_config(page_title="Telco Churn Predictor", page_icon="🔮", layout="wide")
st.title("🔮 Telco Customer Churn Predictor")

# Backend Connection Check
try:
    health_res = requests.get(f"{FASTAPI_URL}/health", timeout=3)
    if health_res.status_code == 200:
        st.success(f"Connected to FastAPI Backend ({FASTAPI_URL})")
    else:
        st.warning("Backend service degraded.")
except Exception as e:
    st.error(f"Cannot connect to FastAPI at {FASTAPI_URL}: {e}")

st.info("Interactive input forms and probability gauge charts will be wired here next.")

Overwriting app.py


In [6]:
%%writefile .github/workflows/ci.yml
name: Telco Churn CI/CD Pipeline

on:
  push:
    branches: [ "main", "master" ]
  pull_request:
    branches: [ "main", "master" ]

jobs:
  test:
    name: Run Unit Tests
    runs-on: ubuntu-latest

    steps:
      - name: Checkout Code
        uses: actions/checkout@v4

      - name: Set up Python 3.11
        uses: actions/setup-python@v5
        with:
          python-version: "3.11"
          cache: "pip"

      - name: Install Dependencies
        run: |
          python -m pip install --upgrade pip
          pip install pytest pytest-cov
          if [ -f requirements.txt ]; then pip install -r requirements.txt; fi

      - name: Run Pytest Suite
        env:
          MLFLOW_ALLOW_FILE_STORE: "true"
          MLFLOW_TRACKING_URI: "file:./mlruns_ci"
          PYTHONPATH: "."
        run: |
          python -m pytest tests/ -v

  docker-build:
    name: Validate Docker Stack Build
    needs: test
    runs-on: ubuntu-latest

    steps:
      - name: Checkout Code
        uses: actions/checkout@v4

      - name: Set up Docker Buildx
        uses: docker/setup-buildx-action@v3

      - name: Build FastAPI Target
        uses: docker/build-push-action@v5
        with:
          context: .
          target: fastapi
          push: false
          tags: telco-fastapi:ci-test

      - name: Build Streamlit Target
        uses: docker/build-push-action@v5
        with:
          context: .
          target: streamlit
          push: false
          tags: telco-streamlit:ci-test

Overwriting .github/workflows/ci.yml


In [5]:
%%writefile requirements.txt
fastapi
uvicorn
streamlit
requests
pydantic
xgboost
mlflow
pandas
numpy
scikit-learn
tabulate
optuna
joblib
pytest
pytest-cov
plotly>=5.18.0

Overwriting requirements.txt


In [5]:
%%writefile .gitignore
__pycache__/
*.pyc
.pytest_cache/
.venv/
env/
venv/
mlruns/
.ipynb_checkpoints/
artifacts/*.pkl

Writing .gitignore


In [1]:
%%writefile app.py
import os
import plotly.graph_objects as go
import requests
import streamlit as st

# Streamlit Page Configuration
st.set_page_config(
    page_title="Telco Churn Predictor",
    page_icon="🔮",
    layout="wide",
    initial_sidebar_state="expanded",
)

# Configurable FastAPI Endpoint (Supports Docker Compose & Local Setup)
BACKEND_URL = os.getenv("BACKEND_URL", "http://localhost:8000")


def check_backend_health():
    """Verify connectivity to FastAPI inference service."""
    try:
        res = requests.get(f"{BACKEND_URL}/health", timeout=3)
        return res.status_code == 200, res.json()
    except Exception:
        return False, {}


# App Header
st.title("🔮 Telco Customer Churn Predictor")
st.markdown(
    "Evaluate customer churn probability in real time using your trained XGBoost model."
)

# Backend Health Status Indicator
is_healthy, health_data = check_backend_health()
if is_healthy:
    st.sidebar.success(f"🟢 Connected to FastAPI (`{BACKEND_URL}`)")
else:
    st.sidebar.error(
        f"🔴 Backend Disconnected (`{BACKEND_URL}`). Ensure FastAPI container is running."
    )

st.markdown("---")

# Feature Input Form UI
st.subheader("📋 Customer Profile & Service Selection")

col1, col2, col3 = st.columns(3)

with col1:
    st.markdown("#### 👤 Demographics")
    gender = st.selectbox("Gender", ["Female", "Male"])
    senior_citizen = st.selectbox(
        "Senior Citizen", [0, 1], format_func=lambda x: "Yes" if x == 1 else "No"
    )
    partner = st.selectbox("Partner", ["Yes", "No"])
    dependents = st.selectbox("Dependents", ["Yes", "No"])
    tenure = st.slider("Tenure (Months)", min_value=1, max_value=72, value=12)

with col2:
    st.markdown("#### 🌐 Subscribed Services")
    internet_service = st.selectbox(
        "Internet Service", ["Fiber optic", "DSL", "No"]
    )
    online_security = st.selectbox(
        "Online Security", ["No", "Yes", "No internet service"]
    )
    online_backup = st.selectbox(
        "Online Backup", ["No", "Yes", "No internet service"]
    )
    device_protection = st.selectbox(
        "Device Protection", ["No", "Yes", "No internet service"]
    )
    tech_support = st.selectbox(
        "Tech Support", ["No", "Yes", "No internet service"]
    )
    streaming_tv = st.selectbox(
        "Streaming TV", ["No", "Yes", "No internet service"]
    )
    streaming_movies = st.selectbox(
        "Streaming Movies", ["No", "Yes", "No internet service"]
    )

with col3:
    st.markdown("#### 💳 Account & Financials")
    contract = st.selectbox(
        "Contract Type", ["Month-to-month", "One year", "Two year"]
    )
    paperless_billing = st.selectbox("Paperless Billing", ["Yes", "No"])
    payment_method = st.selectbox(
        "Payment Method",
        [
            "Electronic check",
            "Mailed check",
            "Bank transfer (automatic)",
            "Credit card (automatic)",
        ],
    )
    phone_service = st.selectbox("Phone Service", ["Yes", "No"])
    multiple_lines = st.selectbox(
        "Multiple Lines", ["No", "Yes", "No phone service"]
    )
    monthly_charges = st.number_input(
        "Monthly Charges ($)",
        min_value=18.0,
        max_value=150.0,
        value=70.0,
        step=1.0,
    )
    total_charges = st.number_input(
        "Total Charges ($)",
        min_value=18.0,
        max_value=9000.0,
        value=float(tenure * monthly_charges),
        step=10.0,
    )

# Format JSON Payload matching CustomerPayload Schema
payload = {
    "gender": gender,
    "SeniorCitizen": senior_citizen,
    "Partner": partner,
    "Dependents": dependents,
    "tenure": tenure,
    "PhoneService": phone_service,
    "MultipleLines": multiple_lines,
    "InternetService": internet_service,
    "OnlineSecurity": online_security,
    "OnlineBackup": online_backup,
    "DeviceProtection": device_protection,
    "TechSupport": tech_support,
    "StreamingTV": streaming_tv,
    "StreamingMovies": streaming_movies,
    "Contract": contract,
    "PaperlessBilling": paperless_billing,
    "PaymentMethod": payment_method,
    "MonthlyCharges": monthly_charges,
    "TotalCharges": total_charges,
}

st.markdown("---")

# Submit & Predict
predict_btn = st.button("🚀 Calculate Churn Probability", use_container_width=True)

if predict_btn:
    if not is_healthy:
        st.error("Cannot complete prediction: Backend server is currently offline.")
    else:
        with st.spinner("Analyzing risk profile..."):
            try:
                response = requests.post(
                    f"{BACKEND_URL}/predict", json=payload, timeout=5
                )

                if response.status_code == 200:
                    result = response.json()
                    probability = float(result.get("churn_probability", 0.0))
                    prediction = int(result.get("churn_prediction", 0))

                    st.markdown("## 📊 Model Inference Results")

                    res_col1, res_col2 = st.columns([1, 1])

                    with res_col1:
                        # Gauge Chart Visualization
                        fig = go.Figure(
                            go.Indicator(
                                mode="gauge+number",
                                value=probability * 100,
                                number={"suffix": "%", "font": {"size": 36}},
                                title={"text": "Churn Risk Probability"},
                                gauge={
                                    "axis": {"range": [0, 100]},
                                    "bar": {"color": "#333333"},
                                    "steps": [
                                        {"range": [0, 30], "color": "#2ecc71"},
                                        {"range": [30, 60], "color": "#f1c40f"},
                                        {"range": [60, 100], "color": "#e74c3c"},
                                    ],
                                    "threshold": {
                                        "line": {"color": "black", "width": 4},
                                        "thickness": 0.75,
                                        "value": probability * 100,
                                    },
                                },
                            )
                        )
                        fig.update_layout(
                            height=280, margin=dict(l=20, r=20, t=40, b=20)
                        )
                        st.plotly_chart(fig, use_container_width=True)

                    with res_col2:
                        st.markdown("### Risk Status & Strategy")
                        if prediction == 1 or probability >= 0.5:
                            st.error("⚠️ **HIGH RISK CUSTOMER**")
                            st.write(
                                "This customer profile shows strong indicators of potential cancellation."
                            )
                            st.markdown(
                                """
                                **Recommended Actions:**
                                * Offer annual contract conversion discounts.
                                * Upgrade internet speed or add complimentary tech support.
                                """
                            )
                        else:
                            st.success("✅ **LOW RISK CUSTOMER**")
                            st.write(
                                "Customer displays stable usage patterns and low risk profile."
                            )
                            st.markdown(
                                """
                                **Recommended Actions:**
                                * Cross-sell higher tier value bundles.
                                * Invite to loyalty feedback program.
                                """
                            )
                else:
                    st.error(f"Inference Error ({response.status_code}): {response.text}")

            except Exception as err:
                st.error(f"Failed to communicate with prediction service: {err}")

Overwriting app.py


SyntaxError: invalid syntax (890191187.py, line 1)